# 🧩 4 Retos de Equipo: Optimizadores en Deep Learning
## SGD, Momentum, Adam y el gradiente que desaparece — puestos a prueba

---

### 🎯 Cómo funciona esta clase

La clase se divide en **4 equipos de 5 integrantes**. Cada equipo recibe **un único reto** (no los cuatro) y tiene **45 minutos** para resolverlo en conjunto, discutiendo antes de programar. Al final, cada equipo comparte sus hallazgos con el resto de la clase en una puesta en común de 3-5 minutos por equipo.

| Equipo | Reto | Tema central |
|---|---|---|
| **Equipo 1** | 🧗 El rescate del cañón | SGD vs. SGD + Momentum |
| **Equipo 2** | ⚡ La central de energía | AdaGrad vs. RMSProp |
| **Equipo 3** | 🔎 El caso del gradiente que desaparece | Vanishing gradient y cómo diagnosticarlo |
| **Equipo 4** | 🗿 El duelo de los escultores | SGD vs. Momentum vs. Adam, en la práctica |

Los cuatro retos comparten una misma pregunta de fondo: **¿cómo decide una red neuronal cuánto y hacia dónde mover sus pesos?** Cada equipo va a llegar a una pieza distinta de esa respuesta — y al juntarlas todas en la puesta en común, la clase entera va a tener el panorama completo.

---

### 👥 Roles sugeridos dentro de cada equipo (5 personas)

1. **Facilitador/a** — guía la discusión, se asegura de que todos opinen antes de escribir código.
2. **Cronometrista** — controla los 45 minutos y avisa a los 15, 30 y 40 minutos.
3. **Programador/a principal** — escribe el código en el notebook, comparte pantalla.
4. **Co-programador/a** — revisa el código en voz alta, detecta errores, propone alternativas.
5. **Presentador/a** — prepara y da la puesta en común final (puede apoyarse en el resto).

*(Roles flexibles: en equipos más chicos, se pueden combinar.)*

---

### ⏱️ Sugerencia de reparto del tiempo (45 min)

- **0–10 min:** Leer el enunciado del reto completo, discutir en voz alta qué está pasando y por qué, antes de tocar el código.
- **10–35 min:** Completar las secciones marcadas con 🛠️ **TU MISIÓN**, ejecutar, iterar.
- **35–45 min:** Responder las preguntas de discusión y preparar 2-3 minutos de puesta en común (qué encontraron, un gráfico, una conclusión).

---

### 🔧 Setup compartido

Antes de que cada equipo salte a su reto, corran esta celda una vez — carga las librerías y el dataset público que usan los Retos 3 y 4.


In [ ]:
# Setup compartido para los 4 retos — ejecutar una sola vez
!pip install -q numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# Dataset público compartido por los Retos 3 y 4: Iris (flores), binario (versicolor vs. virginica)
url_iris = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
iris_full = pd.read_csv(url_iris)
iris = iris_full[iris_full["species"].isin(["versicolor", "virginica"])].reset_index(drop=True)
iris["etiqueta"] = (iris["species"] == "virginica").astype(int)

X_iris = iris[["sepal_length", "sepal_width", "petal_length", "petal_width"]].values.astype(float)
y_iris = iris["etiqueta"].values.astype(float).reshape(-1, 1)
X_iris = (X_iris - X_iris.mean(axis=0)) / X_iris.std(axis=0)

from sklearn.model_selection import train_test_split
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

print("✅ Setup listo. Dataset Iris cargado:", X_iris.shape[0], "flores (versicolor / virginica).")
print("Cada equipo: bajen ahora a la sección de SU reto únicamente.")


---
---
# 🧗 RETO 1 — EQUIPO 1: El rescate del cañón

### 📜 La misión

Hay una bolita atrapada en el fondo de un **cañón angosto y curvo**. Tiene que llegar hasta la salida — el punto más bajo del terreno — pero cada vez que intenta avanzar, choca contra las paredes del cañón y rebota de un lado a otro sin poder avanzar en línea recta.

Ustedes son el **equipo de rescate**. Tienen dos herramientas: un algoritmo llamado **SGD** (el que está usando la bolita ahora mismo y que la tiene atrapada) y un algoritmo mejorado llamado **SGD + Momentum**, que todavía no está terminado de programar.

**Su misión:** entender por qué la bolita está atrapada, terminar de programar Momentum, y demostrar con datos que efectivamente la rescata.

### 🧠 El terreno (para que lo tengan en cuenta)

El "cañón" es un valle muy angosto: empinado en una dirección y muy achatado en la otra — el terreno clásico para poner a prueba si un optimizador zigzaguea o avanza derecho:

$$ f(x,y) = x^2 + 100\,y^2 $$

El punto de salida (mínimo global) está en $(0, 0)$.


In [ ]:
# ============================================================
# PASO 1 — Así está el terreno, y así está atrapada la bolita con SGD
# (Este código YA funciona. Ejecútenlo y observen el gráfico.)
# ============================================================

def valle(x, y):
    return x ** 2 + 100 * y ** 2

def valle_grad(x, y):
    return np.array([2 * x, 200 * y])

def run_sgd(start, lr, n_iter):
    theta = np.array(start, dtype=float)
    path = [theta.copy()]
    for _ in range(n_iter):
        g = valle_grad(*theta)
        theta = theta - lr * g
        path.append(theta.copy())
    return np.array(path)

xs = np.linspace(-9, 2, 300)
ys = np.linspace(-3.5, 3.5, 300)
X, Y = np.meshgrid(xs, ys)
Z = valle(X, Y)

start_point = (-8.0, 3.0)
path_sgd = run_sgd(start_point, lr=0.0099, n_iter=150)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.85)
ax.plot(*path_sgd.T, color="white", lw=1.5, label="SGD (atrapada)")
ax.plot(0, 0, marker="*", color="red", markersize=18, label="Salida (mínimo)")
ax.plot(*start_point, marker="o", color="black", markersize=8, label="La bolita arranca acá")
ax.set_title("La bolita con SGD: ¿llega a la salida, o rebota contra las paredes?")
ax.legend(loc="upper right")
plt.show()

print(f"Pérdida final con SGD: {valle(*path_sgd[-1]):.4f}  (0.0 = llegó a la salida)")


### 💬 Discutan antes de seguir (5 minutos)

- Miren el gráfico: ¿la bolita avanza en línea recta hacia la salida, o zigzaguea?
- ¿Qué pasaría si aumentan `lr` (el tamaño del paso)? ¿Y si lo bajan mucho? *(Pueden probar cambiando el valor en la celda de arriba y volviendo a correr.)*
- Como equipo de rescate, ¿qué le darían a la bolita para que deje de rebotar contra las paredes y avance derecho por el medio del cañón?

### 🛠️ TU MISIÓN

El código de abajo define `run_momentum`, pero **está incompleto**: en vez de acumular una memoria de hacia dónde se movió antes, en cada paso usa el gradiente crudo — por eso todavía se comporta igual que SGD. Tienen que completar la línea marcada con `TODO`.

**Pista:** la fórmula de Momentum es
$$ v_t = \beta \cdot v_{t-1} + (1-\beta) \cdot g_t $$
$$ \theta_{t+1} = \theta_t - \eta \cdot v_t $$


In [ ]:
def run_momentum(start, lr, n_iter, beta=0.9):
    theta = np.array(start, dtype=float)
    v = np.zeros_like(theta)
    path = [theta.copy()]
    for _ in range(n_iter):
        g = valle_grad(*theta)

        # 🛠️ TU MISIÓN: esta línea todavía no usa memoria de los pasos anteriores.
        # Reemplacen la línea de abajo por la fórmula real de Momentum:
        #   v = beta * v + (1 - beta) * g
        v = g  # <-- TODO: arreglar esta línea

        theta = theta - lr * v
        path.append(theta.copy())
    return np.array(path)


path_momentum = run_momentum(start_point, lr=0.0099, n_iter=150, beta=0.9)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.85)
ax.plot(*path_sgd.T, color="gray", lw=1, linestyle="--", label="SGD (atrapada)")
ax.plot(*path_momentum.T, color="orange", lw=1.5, label="SGD + Momentum (su rescate)")
ax.plot(0, 0, marker="*", color="red", markersize=18, label="Salida (mínimo)")
ax.set_title("¿Lograron rescatar a la bolita?")
ax.legend(loc="upper right")
plt.show()

print(f"Pérdida final SGD:      {valle(*path_sgd[-1]):.4f}")
print(f"Pérdida final Momentum: {valle(*path_momentum[-1]):.4f}  (si está mucho más cerca de 0, ¡la rescataron!)")


### 📋 Preguntas para la puesta en común

1. ¿Qué línea de código tuvieron que cambiar, y por qué esa línea es la clave de todo Momentum?
2. En sus propias palabras (sin fórmulas): ¿qué significa que la bolita tenga "memoria" de hacia dónde venía moviéndose?
3. Prueben con `beta=0.5` y `beta=0.99`. ¿Qué cambia? ¿Por qué creen que `beta=0.9` es un valor tan usado en la práctica?
4. Si tuvieran que explicarle Momentum a alguien que nunca vio una red neuronal, ¿qué analogía de la vida cotidiana usarían?


---
---
# ⚡ RETO 2 — EQUIPO 2: La central de energía

### 📜 La misión

Imaginen una central de energía que reparte electricidad a un barrio. Cada casa del barrio es un **parámetro** de la red neuronal. Durante meses, la central le dio a **todas las casas exactamente el mismo voltaje**, sin importar si esa casa consume mucho, poco, todo el tiempo, o solo a veces.

Alguien de la central tuvo una idea — **AdaGrad**: llevar una bitácora de *todo* el consumo histórico de cada casa, y repartir el voltaje según eso. El problema es que la bitácora **nunca se borra, solo crece**, y con el tiempo la central termina bajándole el voltaje a casi todo el barrio hasta casi apagarlo.

**Su misión:** entender por qué la central de AdaGrad se está apagando sola, y terminar de programar la solución — **RMSProp** — que usa una bitácora con "memoria reciente" en vez de un historial que nunca se borra.

### 🧠 El terreno

Un poco más retorcido que el del Equipo 1 — la función de Rosenbrock, un valle angosto *y curvo*, clásico para poner a prueba optimizadores:

$$ f(x,y) = (1-x)^2 + 100(y - x^2)^2 $$


In [ ]:
# ============================================================
# PASO 1 — Así se va apagando la central con AdaGrad
# (Este código YA funciona. Ejecútenlo y observen el gráfico.)
# ============================================================

def rosenbrock(x, y):
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def rosenbrock_grad(x, y):
    dfdx = -2 * (1 - x) - 400 * x * (y - x ** 2)
    dfdy = 200 * (y - x ** 2)
    return np.array([dfdx, dfdy])

def run_adagrad(start, lr, n_iter, eps=1e-8):
    theta = np.array(start, dtype=float)
    grad_sq_sum = np.zeros_like(theta)   # la "bitácora" que nunca se borra
    path = [theta.copy()]
    for _ in range(n_iter):
        g = rosenbrock_grad(*theta)
        grad_sq_sum += g ** 2
        theta = theta - (lr / (np.sqrt(grad_sq_sum) + eps)) * g
        path.append(theta.copy())
    return np.array(path)

xs = np.linspace(-2, 2, 300)
ys = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(xs, ys)
Z = rosenbrock(X, Y)

start_point = (-1.5, 2.0)
path_adagrad = run_adagrad(start_point, lr=0.9, n_iter=800)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.85)
axes[0].plot(*path_adagrad.T, color="white", lw=2, label="AdaGrad")
axes[0].plot(1, 1, marker="*", color="red", markersize=18, label="Salida (mínimo)")
axes[0].set_title("Trayectoria de AdaGrad")
axes[0].legend(loc="upper left")

axes[1].plot([rosenbrock(*p) for p in path_adagrad], color="white" if False else "steelblue")
axes[1].set_yscale("log")
axes[1].set_title("Pérdida vs. iteración — ¿se aplana antes de llegar a 0?")
axes[1].set_xlabel("Iteración")
plt.tight_layout()
plt.show()

print(f"Pérdida final con AdaGrad: {rosenbrock(*path_adagrad[-1]):.4f}  (0.0 = llegó a la salida)")


### 💬 Discutan antes de seguir (5 minutos)

- Miren el segundo gráfico (pérdida vs. iteración). ¿La curva sigue bajando al final, o se aplana antes de llegar a la salida?
- ¿Por qué una bitácora que **solo crece** (`grad_sq_sum += g**2`, nunca se resetea ni se "olvida" de nada) puede terminar apagando el voltaje casi a cero?
- ¿Qué tendría que hacer la central para no perder de vista el consumo "reciente" sin cargar con todo el historial completo desde el día uno?

### 🛠️ TU MISIÓN

El código de `run_rmsprop` de abajo está incompleto: en vez de una **memoria reciente** (promedio móvil), todavía usa una bitácora que se resetea en cada paso y no recuerda nada de un paso a otro — por eso el voltaje que reparte es errático. Completen la línea marcada con `TODO`.

**Pista:** la fórmula de RMSProp es
$$ v_t = \beta \cdot v_{t-1} + (1-\beta) \cdot g_t^2 $$
$$ \theta_{t+1} = \theta_t - \dfrac{\eta}{\sqrt{v_t}+\epsilon} \cdot g_t $$


In [ ]:
def run_rmsprop(start, lr, n_iter, beta=0.9, eps=1e-8):
    theta = np.array(start, dtype=float)
    v = np.zeros_like(theta)
    path = [theta.copy()]
    for _ in range(n_iter):
        g = rosenbrock_grad(*theta)

        # 🛠️ TU MISIÓN: esta línea no acumula memoria de un paso a otro.
        # Reemplacen la línea de abajo por la fórmula real de RMSProp:
        #   v = beta * v + (1 - beta) * (g ** 2)
        v = g ** 2  # <-- TODO: arreglar esta línea

        theta = theta - (lr / (np.sqrt(v) + eps)) * g
        path.append(theta.copy())
    return np.array(path)


path_rmsprop = run_rmsprop(start_point, lr=0.02, n_iter=800, beta=0.9)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.85)
axes[0].plot(*path_adagrad.T, color="gray", lw=1.5, linestyle="--", label="AdaGrad (se apaga)")
axes[0].plot(*path_rmsprop.T, color="orange", lw=2, label="RMSProp (su solución)")
axes[0].plot(1, 1, marker="*", color="red", markersize=18, label="Salida (mínimo)")
axes[0].legend(loc="upper left")
axes[0].set_title("¿La central sigue funcionando hasta el final?")

axes[1].plot([rosenbrock(*p) for p in path_adagrad], color="gray", linestyle="--", label="AdaGrad")
axes[1].plot([rosenbrock(*p) for p in path_rmsprop], color="orange", label="RMSProp")
axes[1].set_yscale("log")
axes[1].legend()
axes[1].set_title("Pérdida vs. iteración")
plt.tight_layout()
plt.show()

print(f"Pérdida final AdaGrad: {rosenbrock(*path_adagrad[-1]):.4f}")
print(f"Pérdida final RMSProp: {rosenbrock(*path_rmsprop[-1]):.4f}")


### 📋 Preguntas para la puesta en común

1. ¿Qué línea de código tuvieron que cambiar, y qué diferencia real hace ese cambio?
2. ¿Por qué RMSProp no sufre el mismo "apagón" que AdaGrad, aunque ambos dividen el paso por una magnitud de gradiente?
3. Prueben con `beta=0.5` y `beta=0.99` en RMSProp. ¿Qué cambia?
4. El Equipo 1 va a hablar de Momentum. ¿En qué se parece su "memoria reciente" (RMSProp) a la memoria de Momentum, y en qué es fundamentalmente distinta?


---
---
# 🔎 RETO 3 — EQUIPO 3: El caso del gradiente que desaparece

### 📜 La misión

Son un equipo de **investigación forense de datos**. Les llega una red neuronal de 5 capas ocultas que se supone tiene que aprender a distinguir dos especies de flores (Iris — versicolor vs. virginica), pero **no está aprendiendo**: la pérdida casi no baja, entrenamiento tras entrenamiento.

Alguien sospecha que el problema está en las **primeras capas** de la red, que parecen estar "congeladas" — como si no recibieran ninguna instrucción para cambiar. Ustedes tienen que investigar la escena, medir la evidencia (la magnitud del gradiente en cada capa), encontrar al culpable, y proponer una solución.

**Su misión:** diagnosticar el problema midiendo el gradiente capa por capa, confirmar la hipótesis del vanishing gradient, e implementar al menos una solución real.


In [ ]:
# ============================================================
# PASO 1 — La escena del crimen: una red de 5 capas sigmoide que no aprende
# (Este código YA funciona. Ejecútenlo y observen los dos gráficos.)
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(a):
    return a * (1 - a)

class MLPInvestigacion:
    def __init__(self, layer_sizes, seed=1, activation="sigmoid"):
        rng = np.random.default_rng(seed)
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.W = []
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            # Escala de inicialización apropiada para cada activación
            # (sigmoide: 0.5 fijo · ReLU: escala de He, sqrt(2/fan_in), para evitar activaciones que explotan)
            scale = np.sqrt(2.0 / fan_in) if activation == "relu" else 0.5
            self.W.append(rng.normal(0, scale, size=(fan_in, layer_sizes[i+1])))
        self.b = [np.zeros((1, layer_sizes[i+1])) for i in range(len(layer_sizes)-1)]

    def act(self, z):
        if self.activation == "sigmoid":
            return sigmoid(z)
        else:
            # 🛠️ TU MISIÓN (parte 2): esta rama todavía no es una ReLU real.
            # Reemplacen la línea de abajo por una ReLU real: np.maximum(0, z)
            return sigmoid(z)  # <-- TODO: arreglar esta línea para que sea ReLU real

    def act_deriv(self, a, z):
        if self.activation == "sigmoid":
            return sigmoid_deriv(a)
        else:
            # 🛠️ TU MISIÓN (parte 2): derivada de ReLU real es 1 donde z>0, si no 0.
            return sigmoid_deriv(a)  # <-- TODO: arreglar esta línea para que sea derivada de ReLU

    def forward(self, X):
        activations = [X]
        zs = []
        a = X
        for W, b in zip(self.W, self.b):
            z = a @ W + b
            a = self.act(z)
            zs.append(z)
            activations.append(a)
        return activations, zs

    def backward(self, activations, zs, y):
        n = y.shape[0]
        grads_W = [None] * len(self.W)
        grad_norms = [None] * len(self.W)
        y_hat = activations[-1]
        delta = (y_hat - y) * sigmoid_deriv(y_hat)  # capa de salida siempre sigmoide (clasificación binaria)
        for l in reversed(range(len(self.W))):
            a_prev = activations[l]
            grads_W[l] = (a_prev.T @ delta) / n
            grad_norms[l] = np.linalg.norm(grads_W[l])
            if l > 0:
                delta = (delta @ self.W[l].T) * self.act_deriv(activations[l], zs[l-1])
        return grads_W, grad_norms

    def update(self, grads_W, lr):
        for l in range(len(self.W)):
            self.W[l] -= lr * grads_W[l]

    def bce_loss(self, y_hat, y, eps=1e-9):
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))


def investigar(activation, epochs=300, lr=0.5):
    arquitectura = [X_train_iris.shape[1], 16, 16, 16, 16, 16, 1]
    modelo = MLPInvestigacion(arquitectura, seed=1, activation=activation)
    loss_hist, grad_first_hist, grad_last_hist = [], [], []
    for _ in range(epochs):
        activations, zs = modelo.forward(X_train_iris)
        loss = modelo.bce_loss(activations[-1], y_train_iris)
        grads_W, grad_norms = modelo.backward(activations, zs, y_train_iris)
        modelo.update(grads_W, lr)
        loss_hist.append(loss)
        grad_first_hist.append(grad_norms[0])
        grad_last_hist.append(grad_norms[-1])
    return {"loss": loss_hist, "grad_first": grad_first_hist, "grad_last": grad_last_hist}


evidencia_sigmoid = investigar(activation="sigmoid", epochs=300, lr=0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(evidencia_sigmoid["loss"], color="steelblue")
axes[0].set_title("Pérdida durante el entrenamiento (todo sigmoide)")
axes[0].set_xlabel("Época")

axes[1].plot(evidencia_sigmoid["grad_first"], label="Capa 1 (temprana)", color="crimson")
axes[1].plot(evidencia_sigmoid["grad_last"], label="Última capa", color="seagreen")
axes[1].set_yscale("log")
axes[1].set_title("Evidencia clave: norma del gradiente por capa")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Pérdida final: {evidencia_sigmoid['loss'][-1]:.4f}")


### 💬 Discutan antes de seguir (5 minutos)

- Miren el segundo gráfico: ¿la curva roja (capa 1) está muy por debajo de la verde (última capa)? ¿Qué tan por debajo — el doble, 10 veces menos, 100 veces menos?
- Si la capa 1 recibe un gradiente casi nulo, ¿qué le pasa a sus pesos durante el entrenamiento?
- ¿Por qué creen que el problema es peor en la capa 1 (la más lejana de la salida) y no en la última capa?

### 🛠️ TU MISIÓN

Arriba, la clase `MLPInvestigacion` tiene una opción `activation="relu"`, pero **todavía no está implementada de verdad** — por ahora, aunque le pidan ReLU, sigue usando sigmoide por dentro. Busquen los dos `TODO` dentro de la clase (en `act` y en `act_deriv`) y complétenlos con la ReLU real:

- $\text{ReLU}(z) = \max(0, z)$
- $\text{ReLU}'(z) = 1$ si $z>0$, si no $0$

Después, corran la celda de abajo para comparar.


In [ ]:
# Una vez que hayan completado los TODO de ReLU en la clase de arriba,
# vuelvan a correr esa celda y después esta para comparar sigmoide vs. ReLU.

evidencia_relu = investigar(activation="relu", epochs=300, lr=0.01)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(evidencia_sigmoid["loss"], label="Todo sigmoide", color="crimson")
axes[0].plot(evidencia_relu["loss"], label="Con ReLU", color="seagreen")
axes[0].set_title("Pérdida: ¿resolvieron el caso?")
axes[0].set_xlabel("Época"); axes[0].legend()

axes[1].plot(evidencia_sigmoid["grad_first"], label="Capa 1 — sigmoide", color="crimson", linestyle="--")
axes[1].plot(evidencia_relu["grad_first"], label="Capa 1 — ReLU", color="seagreen", linestyle="--")
axes[1].set_yscale("log")
axes[1].set_title("Gradiente en la capa 1: antes vs. después")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Pérdida final sigmoide: {evidencia_sigmoid['loss'][-1]:.4f}")
print(f"Pérdida final ReLU:     {evidencia_relu['loss'][-1]:.4f}")


### 📋 Preguntas para la puesta en común

1. ¿Cuál fue la "evidencia clave" que les permitió confirmar que el problema era vanishing gradient, y no otra cosa (por ejemplo, un learning rate mal elegido)?
2. ¿Por qué cambiar la activación a ReLU cambia tanto la magnitud del gradiente en la capa 1?
3. Ahora que "resolvieron el caso" con ReLU, ¿alguna otra solución del laboratorio principal (inicialización, batch normalization, conexiones residuales, Adam) también podría haber ayudado? ¿Por qué?
4. Si tuvieran que explicarle a alguien sin experiencia técnica por qué "una red no aprende", ¿cómo resumirían el caso en 2 frases?


---
---
# 🗿 RETO 4 — EQUIPO 4: El duelo de los escultores

### 📜 La misión

Tres aprendices de escultor — **SGD**, **Momentum** y **Adam** — compiten para tallar la misma pieza a partir del mismo bloque: aprender a distinguir dos especies de flores (Iris — versicolor vs. virginica) usando la misma red, los mismos datos, y **el mismo número de golpes de cincel** (épocas de entrenamiento).

Ustedes son el **jurado del duelo**. Tienen que entrenar a los tres aprendices bajo las mismas condiciones, medir su desempeño, y dar un veredicto: ¿quién talló mejor la pieza en el tiempo disponible, y por qué?

### 🧠 Herramienta para el duelo

Para que el duelo sea rápido de correr en los 45 minutos, van a usar `MLPClassifier` de scikit-learn — una red neuronal ya implementada, a la que solo le tienen que cambiar el `solver` (SGD, o Adam) y, en el caso de SGD, el parámetro `momentum`.


In [ ]:
# ============================================================
# PASO 1 — Función para entrenar y medir a un aprendiz
# (Este código YA funciona. Ejecútenlo primero para ver un ejemplo con SGD puro.)
# ============================================================
from sklearn.neural_network import MLPClassifier
import time

def entrenar_aprendiz(nombre, solver, momentum, epochs=200, lr=0.01):
    modelo = MLPClassifier(
        hidden_layer_sizes=(16, 16, 16),
        activation="relu",
        solver=solver,
        momentum=momentum,
        learning_rate_init=lr,
        max_iter=epochs,
        random_state=1,
        n_iter_no_change=epochs,  # para que no pare antes de tiempo y todos compitan con el mismo # de golpes
    )
    t0 = time.time()
    modelo.fit(X_train_iris, y_train_iris.ravel())
    tiempo = time.time() - t0
    acc_train = modelo.score(X_train_iris, y_train_iris.ravel())
    acc_test = modelo.score(X_test_iris, y_test_iris.ravel())
    print(f"{nombre:25s} | acc train: {acc_train:.2%} | acc test: {acc_test:.2%} | "
          f"pérdida final: {modelo.loss_:.4f} | tiempo: {tiempo:.2f}s")
    return modelo

modelo_sgd = entrenar_aprendiz("SGD puro", solver="sgd", momentum=0.0)


### 🛠️ TU MISIÓN

Completen el duelo entrenando también a los otros dos aprendices — **Momentum** y **Adam** — y comparen las tres curvas de pérdida en un mismo gráfico. Reemplacen los `None` de abajo por la llamada a `entrenar_aprendiz(...)` con los parámetros correctos.

**Pistas:**
- Momentum en scikit-learn se activa dentro del mismo solver `"sgd"`, poniendo `momentum=0.9` (en vez de `0.0`).
- Adam es un solver aparte: `solver="adam"` (el parámetro `momentum` se ignora para ese solver, pueden dejarlo en `0.0`).


In [ ]:
# 🛠️ TU MISIÓN: completar estas dos líneas para entrenar a los otros dos aprendices
modelo_momentum = None  # <-- TODO: entrenar_aprendiz("SGD + Momentum", solver="sgd", momentum=0.9)
modelo_adam = None      # <-- TODO: entrenar_aprendiz("Adam", solver="adam", momentum=0.0)

# Una vez completado lo de arriba, esta celda grafica el duelo completo
if modelo_momentum is not None and modelo_adam is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(modelo_sgd.loss_curve_, label="SGD puro", color="#ff6b6b")
    ax.plot(modelo_momentum.loss_curve_, label="SGD + Momentum", color="#8c7bff")
    ax.plot(modelo_adam.loss_curve_, label="Adam", color="#f5c542")
    ax.set_title("El duelo de los escultores: pérdida por época")
    ax.set_xlabel("Época (golpe de cincel)"); ax.set_ylabel("Pérdida")
    ax.legend()
    plt.show()
else:
    print("⚠️ Todavía falta completar los dos TODO de arriba para que el duelo corra completo.")


### 📋 Preguntas para la puesta en común

1. Según su gráfico: ¿quién llegó primero a una pérdida baja? ¿Alguno se quedó claramente atrás con el mismo número de épocas?
2. Miren el `acc test` de cada aprendiz impreso en el Paso 1 (y el que impriman ustedes para Momentum y Adam). ¿El que ganó en velocidad de entrenamiento también ganó en generalización?
3. Prueben cambiar `epochs` a un número bien chico (por ejemplo 20). ¿Cambia el veredicto del duelo? ¿Por qué eso tiene sentido con lo que sabemos de cada optimizador?
4. Si el "bloque de mármol" fuera mucho más difícil (una red mucho más profunda, o un dataset más complicado), ¿esperarían que la diferencia entre los tres aprendices se achique o se agrande? ¿Por qué?


---
---
# 🎤 Puesta en común (después de los 45 minutos)

Cada equipo tiene **3-5 minutos** para compartir con la clase:

1. Una frase que resuma su reto (sin fórmulas — como si se lo explicaran a un compañero de otro equipo).
2. Un gráfico o número que muestre su hallazgo principal.
3. Una respuesta a esta pregunta común para los cuatro equipos:

> **¿Qué tienen en común, en el fondo, Momentum, RMSProp y Adam? ¿Y en qué se diferencian entre sí?**

*(Pista para el debate final: los cuatro retos son, en realidad, cuatro caras del mismo problema — cómo decidir cuánto y hacia dónde mover los pesos de una red. Momentum resuelve la dirección. RMSProp resuelve la escala. Adam junta las dos. Y el vanishing gradient es el recordatorio de que ni el mejor optimizador puede compensar del todo una arquitectura mal diseñada.)*
